# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
# Load starter dataset
if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/hafizahmadadilaiengineer/flyrank-ml-internship.git

repo_root = Path("flyrank-ml-internship")

df = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")



print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("Unique content pages:", df["content_id"].nunique())

Dataset shape: (30000, 44)
Unique clients: 32
Unique content pages: 30000


In [26]:
# Load the baseline queue created during Week 4

baseline_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"

print("Looking for:", baseline_path)
print("Exists:", baseline_path.exists())

if not baseline_path.exists():
    raise FileNotFoundError(
        f"Baseline queue not found at: {baseline_path}"
    )

baseline_queue = pd.read_csv(baseline_path)

print("Baseline queue shape:", baseline_queue.shape)
print("Baseline columns:", list(baseline_queue.columns))

display(baseline_queue.head(10))

Looking for: flyrank-ml-internship/work/outputs/baseline_action_score.csv
Exists: True
Baseline queue shape: (30000, 4)
Baseline columns: ['content_id', 'baseline_score', 'reason_code', 'action']


,content_id,baseline_score,reason_code,action
0,content_6226ee6adc91,100,low_ctr_visible_page,Refresh Immediately
1,content_fe16a55cd13d,100,low_ctr_visible_page,Refresh Immediately
2,content_cf56e2e2e282,100,low_ctr_visible_page,Refresh Immediately
3,content_e3ff1b093148,100,low_ctr_visible_page,Refresh Immediately
4,content_c2d929d83eaa,100,low_ctr_visible_page,Refresh Immediately
5,content_72496874f806,100,low_ctr_visible_page,Refresh Immediately
6,content_0a91db491d14,100,low_ctr_visible_page,Refresh Immediately
7,content_928af3e22c80,100,low_ctr_visible_page,Refresh Immediately
8,content_77d4d5930e5e,100,low_ctr_visible_page,Refresh Immediately
9,content_7f116ae1f6f5,100,low_ctr_visible_page,Refresh Immediately


### How the ranked queue should be used

The Week-4 baseline is used as the action-ranking starting point because it achieved Precision@50 of 0.72 in the Week-5 comparison, compared with 0.46 for Random Forest on the same top-50 decision.

The queue is therefore a prioritization tool rather than an automatic decision system.

Each recommendation has a score, a reason code, and an action label so that a human reviewer can understand why a page appears in the queue.

The reason code describes the signal pattern behind the recommendation. It does not establish that the recommended action will improve performance.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [27]:
# Section 1 — Ranked actions + reason codes

action_queue = baseline_queue.copy()

# Add rank based on the existing baseline ordering
action_queue.insert(
    0,
    "rank",
    range(1, len(action_queue) + 1)
)

# Use human-review-oriented action language
action_queue["action"] = action_queue["action"].replace({
    "Refresh Immediately": "Refresh candidate"
})

print("Action queue shape:", action_queue.shape)

print("\nReason codes:")
print(action_queue["reason_code"].value_counts())

print("\nActions:")
print(action_queue["action"].value_counts())

print("\nTop 10 ranked actions:")
display(action_queue.head(10))

Action queue shape: (30000, 5)

Reason codes:
reason_code
low_ctr_visible_page    17756
general_review          12237
stale_visible_page          7
Name: count, dtype: int64

Actions:
action
Review Soon          16564
Monitor              13334
Refresh candidate      102
Name: count, dtype: int64

Top 10 ranked actions:


,rank,content_id,baseline_score,reason_code,action
0,1,content_6226ee6adc91,100,low_ctr_visible_page,Refresh candidate
1,2,content_fe16a55cd13d,100,low_ctr_visible_page,Refresh candidate
2,3,content_cf56e2e2e282,100,low_ctr_visible_page,Refresh candidate
3,4,content_e3ff1b093148,100,low_ctr_visible_page,Refresh candidate
4,5,content_c2d929d83eaa,100,low_ctr_visible_page,Refresh candidate
5,6,content_72496874f806,100,low_ctr_visible_page,Refresh candidate
6,7,content_0a91db491d14,100,low_ctr_visible_page,Refresh candidate
7,8,content_928af3e22c80,100,low_ctr_visible_page,Refresh candidate
8,9,content_77d4d5930e5e,100,low_ctr_visible_page,Refresh candidate
9,10,content_7f116ae1f6f5,100,low_ctr_visible_page,Refresh candidate


### Ranked action logic

The action queue starts from the Week-4 baseline score because it was the strongest method for the top-50 prioritization decision in my Week-5 comparison.

The baseline achieved Precision@50 of 0.72, compared with 0.46 for the Random Forest. Therefore, the more complex model is not automatically preferred for this decision.

Each row contains a rank, score, reason code, and action label. The reason code provides a concise explanation of why the page was prioritized.

The action label is intentionally written as "Refresh candidate" rather than an automatic instruction. The queue is designed to help a human content reviewer decide which pages deserve attention first.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [28]:
intended_use = {
    "primary_user": "Content/editorial team",
    "purpose": "Prioritize pages for human review",
    "decision": "Which pages should be reviewed first?",
    "output": "Ranked action queue with reason codes",
    "automation_level": "Decision support only"
}

for key, value in intended_use.items():
    print(f"{key}: {value}")

primary_user: Content/editorial team
purpose: Prioritize pages for human review
decision: Which pages should be reviewed first?
output: Ranked action queue with reason codes
automation_level: Decision support only


### Intended use

The playbook is intended for a content or SEO team that needs to prioritize limited review time.

It ranks pages using observed search/content signals and provides a reason code and suggested action for human review.

The score is a prioritization signal, not a guarantee that a page is declining or that a particular intervention will improve performance.

### Limits

This work uses the anonymized starter dataset and the current `is_declining_label` proxy.

The label is derived from the current trend signal rather than a future observed outcome. Therefore, the queue should not be described as a proven predictor of future decline.

The results are measured on the available evaluation data and should be treated as directional decision-support evidence.

The system does not explain Google's ranking algorithm and does not establish that refreshing a page causes recovery.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [29]:
review_rules = pd.DataFrame({
    "Before action": [
        "Check whether the page is actually outdated",
        "Check whether search intent has changed",
        "Review the current SERP/search context",
        "Check whether the content has a genuine quality gap",
        "Consider whether the recommended action is appropriate",
        "Record the human decision and reason for disagreement"
    ]
})

display(review_rules)

,Before action
0,Check whether the page is actually outdated
1,Check whether search intent has changed
2,Review the current SERP/search context
3,Check whether the content has a genuine qualit...
4,Consider whether the recommended action is app...
5,Record the human decision and reason for disag...


### Human review rules

A recommendation should be treated as a review priority, not as an instruction.

Before taking action, a human reviewer should verify the page context, search intent, content quality, freshness, and business relevance.

### No-go list

The system should **not** automatically:

- rewrite or publish content;
- delete or prune a page;
- change a URL;
- change canonicalization;
- merge pages;
- change search intent;
- declare that a page will recover;
- claim a Google ranking-factor explanation;
- override editorial judgment.

The model output is limited to prioritization and decision support.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [30]:
monitoring_triggers = pd.DataFrame({
    "Trigger": [
        "Precision@50 deteriorates on a new evaluation sample",
        "Observed declining rate changes materially",
        "Human reviewers frequently reject recommendations",
        "Input feature distributions change substantially",
        "Data definitions or measurement systems change",
        "The current proxy label no longer represents the decision"
    ],
    "Response": [
        "Re-evaluate the baseline and model",
        "Investigate distribution or data changes",
        "Audit reason codes and ranking signals",
        "Review feature stability",
        "Rebuild the data contract and validation",
        "Redefine the target before retraining"
    ]
})

display(monitoring_triggers)

,Trigger,Response
0,Precision@50 deteriorates on a new evaluation ...,Re-evaluate the baseline and model
1,Observed declining rate changes materially,Investigate distribution or data changes
2,Human reviewers frequently reject recommendations,Audit reason codes and ranking signals
3,Input feature distributions change substantially,Review feature stability
4,Data definitions or measurement systems change,Rebuild the data contract and validation
5,The current proxy label no longer represents t...,Redefine the target before retraining


### Monitoring approach

The playbook should not be treated as permanently valid.

The main monitoring signal is whether the ranked queue continues to support the intended review decision. Precision@50 is particularly useful because the team's capacity is represented by the top-50 queue.

Retraining should not happen simply because time has passed. It should be considered when new comparable data is available and there is evidence that the current ranking performance or data relationships have changed.

Any retraining should repeat the client-grouped validation design used in the audit.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [31]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

paper_queue = baseline_queue.copy()

# Add rank if not already present
if "rank" not in paper_queue.columns:
    paper_queue.insert(
        0,
        "rank",
        range(1, len(paper_queue) + 1)
    )

# Make action language human-review oriented
if "action" in paper_queue.columns:
    paper_queue["action"] = paper_queue["action"].replace({
        "Refresh Immediately": "Refresh candidate"
    })

paper_queue_path = output_dir / "action_playbook_queue.csv"

paper_queue.to_csv(
    paper_queue_path,
    index=False
)

print("Saved:", paper_queue_path)
print("Exists:", paper_queue_path.exists())
print("Rows:", len(paper_queue))

display(paper_queue.head(10))

Saved: work/outputs/action_playbook_queue.csv
Exists: True
Rows: 30000


,rank,content_id,baseline_score,reason_code,action
0,1,content_6226ee6adc91,100,low_ctr_visible_page,Refresh candidate
1,2,content_fe16a55cd13d,100,low_ctr_visible_page,Refresh candidate
2,3,content_cf56e2e2e282,100,low_ctr_visible_page,Refresh candidate
3,4,content_e3ff1b093148,100,low_ctr_visible_page,Refresh candidate
4,5,content_c2d929d83eaa,100,low_ctr_visible_page,Refresh candidate
5,6,content_72496874f806,100,low_ctr_visible_page,Refresh candidate
6,7,content_0a91db491d14,100,low_ctr_visible_page,Refresh candidate
7,8,content_928af3e22c80,100,low_ctr_visible_page,Refresh candidate
8,9,content_77d4d5930e5e,100,low_ctr_visible_page,Refresh candidate
9,10,content_7f116ae1f6f5,100,low_ctr_visible_page,Refresh candidate


## Self-check

- [x] Every section above is filled with both reasoning and supporting code.
- [x] The notebook runs top to bottom with no errors.
- [x] The ranked queue uses reason codes and human-readable actions.
- [x] The intended user and decision are clearly stated.
- [x] Human review is required before action.
- [x] Automated no-go actions are explicitly listed.
- [x] Monitoring and retrain triggers are defined.
- [x] The queue is exported to `work/outputs/`.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful observed, measured, directional, and decision-support language.
- [x] The notebook is committed to `work/notebooks/w07_action_playbook.ipynb`.